In [ ]:
# !pip install sklearn tqdm

In [ ]:
import numpy as np
import scipy as sp
import sklearn
import sklearn.linear_model
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
np.random.seed(42)
torch.manual_seed(42);

In [ ]:
# to zoom in the plot
# %matplotlib widget 

# Inverse dynamical model of a mechanical system

## Dataset

We consider the dataset available here: https://www.nonlinearbenchmark.org/benchmarks/emps. 

The test device is a single prismatic joint, which is modeled as a 1-DoF mechanical system of mass $M$. We apply a force $\tau(t)$ (N) and measure the joint position $q(t)$ (m). 
The applied force is actually measured up to an unknown constant bias $b$ (N). Moreover, friction forces $\tau_{f}(t)$ act on the system.

Newton's law suggests:

\begin{equation}
M \ddot q(t) = \tau(t) - \tau_{f}(t) - b
\end{equation}


In this notebook, we illustrate the inverse dynamical modeling of the system, namely the problem of reconstructiong $\tau(t)$ from $q(t)$ and (its derivatives). 

The original dataset contains measurements of $q(t)$ and $\tau(t)$ and is already split in a train and a test part. A few pre-processing steps have been performed in the notebook dataset_preproc.ipynb.
For instance, $\dot q(t)$ and $\ddot q(t)$ have been obtained through a careful numerical differentiation of $q(t)$. 

Let us first explore the data!

In [ ]:
# Load data
data = sp.io.loadmat('EMPS.mat')
train_tau = data['train_tau'] # force (N)
train_q = data['train_q'] # position (m)
train_dq = data['train_dq'] # velocity (m/s)
train_ddq = data['train_ddq'] # train acceleration (m/s^2)
test_tau = data['test_tau']
test_q = data['test_q']
test_dq = data['test_dq']
test_ddq = data['test_ddq']
ts = data['ts'][0, 0]

In [ ]:
train_t = np.arange(0, len(train_q)) * ts
test_t = np.arange(0, len(test_q)) * ts
train_t.shape, test_t.shape

In [ ]:
# Visualize data

fig, ax = plt.subplots(4, 1, figsize=(10, 10))
ax[0].plot(train_t, train_q)
ax[0].set_title(r'train $q$')
ax[0].set_ylabel('Position (m)')
ax[1].plot(train_t, train_dq)
ax[1].set_title(r'train $\dot q$')
ax[1].set_ylabel('Velocity (m/s)')
ax[2].plot(train_t, train_ddq)
ax[2].set_title(r'train $\ddot q$')
ax[2].set_ylabel('Acceleration (m/s^2)')
ax[3].plot(train_t, train_tau)
ax[3].set_title(r'train $\tau$')
ax[3].set_ylabel('Force (N)')
ax[3].set_xlabel('Time (s)');

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(10, 10))
ax[0].plot(test_t, test_q)
ax[0].set_title('test $q$')
ax[0].set_ylabel('Position (m)')
ax[1].plot(test_t, test_dq)
ax[1].set_title(r'test $\dot q$')
ax[1].set_ylabel('Velocity (m/s)')
ax[2].plot(test_t, test_ddq)
ax[2].set_title(r'test $\ddot q$')
ax[2].set_ylabel(r'Acceleration (m/s^2)')
ax[3].plot(test_t, test_tau)
ax[3].set_title(r'test $\tau$')
ax[3].set_ylabel('Force (N)')
ax[3].set_xlabel('Time (s)');

## Linear least-squares IDM

A very simple inverse dynamic model (IDM) for this system is:

$$\tau(t) = M\ddot q(t) + F_v \dot q(t) + b$$

which assumes a friction model:
$$ \tau_{f} = F_v \dot q(t)$$

The free parameters are:
* $M$: system mass
* $F_v$: viscous friction
* $b$: an additional constant, which takes into accound a bias in the measurement setup


The IDM can be written as a standard linear model:

$$ \tau(t) = \phi(t) \theta, \qquad \phi(t) = [\ddot q(t)\, \dot q(t)\, 1], \qquad \theta = [M\, F_v\,  b]^\top$$

The parameters $\theta$ can be fitted with standard least squares.

In [ ]:
# Linear Least squares fitting

train_X = np.c_[train_ddq, train_dq] #, np.ones_like(train_q)]
test_X = np.c_[test_ddq, test_dq] #, np.ones_like(test_q)]

train_y = train_tau
test_y = test_tau

In [ ]:
train_X.shape, train_y.shape

In [ ]:
lin = sklearn.linear_model.LinearRegression().fit(train_X, train_y)
lin.coef_, lin.intercept_

In [ ]:
# Model evaluation
train_y_hat = lin.predict(train_X)
train_rmse = np.sqrt(np.mean((train_y - train_y_hat)**2))
test_y_hat = lin.predict(test_X)
test_rmse = np.sqrt(np.mean((test_y - test_y_hat)**2))
train_rmse, test_rmse

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
plt.suptitle('Linear IDM')
ax[0].plot(train_t, train_y, "k", label='true')
ax[0].plot(train_t, train_y_hat, "b", label='predicted')
ax[0].plot(train_t, train_y - train_y_hat, "r", label='prediction error')
ax[0].legend(loc="upper right")
ax[0].set_title('train data')
ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('Force (N)')

ax[1].plot(test_t, test_y, "k", label='true')
ax[1].plot(test_t, test_y_hat, "b", label='predicted')
ax[1].plot(test_t, test_y - test_y_hat, "r", label='prediction error')
ax[1].legend(loc="upper right")
ax[1].set_title('test data')
ax[1].set_xlabel('Time (s)')
ax[1].set_ylabel('Force (N)');

Same in PyTorch (useful only for educational purpose)

In [ ]:
# Trining data as pytorch tensors
train_X_torch = torch.tensor(np.c_[train_ddq, train_dq]).float()
train_y_torch = torch.tensor(train_tau).float()

In [ ]:
train_X_torch.shape, train_y_torch.shape

In [ ]:
# Linear model in pytorch
friction_lin = nn.Linear(2, 1, bias=True)
friction_lin

In [ ]:
friction_lin.weight.data, friction_lin.bias.data

In [ ]:
# Training hyperparameters
iters = 100_000
batch_size = 64 # minibatch
lr = 1e-2

In [ ]:
# Train the linear pytorch model

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(friction_lin.parameters(), lr=lr)

LOSS = []
for iter in (pbar := tqdm(range(iters))):
    optimizer.zero_grad()
    batch_idx = torch.randint(0, len(train_X_torch), (batch_size,))
    preds = friction_lin(train_X_torch[batch_idx])
    loss = criterion(preds, train_y_torch[batch_idx])
    loss.backward()
    optimizer.step()
    LOSS.append(loss.item())
    if iter % 1000 == 0:
        pbar.set_postfix({'loss': loss.item()})


In [ ]:
friction_lin.weight.data, friction_lin.bias.data # more or less the same as linear least squares

In [ ]:
plt.figure()
plt.plot(LOSS)

## Linear least-squares IDM with static friction


The linear prediction is not too bad, but it looks somewhat *biased*. Actually, the previous IDM model misses **static friction**, which is relevant in this dataset.

Let us enrich the friction model to:
$$ \tau_{f}(t) = F_v \dot q(t) + F_c \text{sign}(\dot q(t))$$

We then have:
$$ \tau(t) = M\ddot q(t) + F_v \dot q(t) + F_c \text{sign}(\dot q(t)) + b$$

The IDM with static friction can be modified to:

$$ \tau(t) = \phi(t) \theta, \qquad \phi(t) = [\ddot q(t)\, \dot q(t)\, \text{sign}(q(t))\, 1], \qquad \theta = [M\, F_v\, F_c\, b]^\top,$$

which can still be treated as a linear regression adding the feature $\text{sign}(q(t))$. In machine-learning terms, we are doing *feature engineering*.

In [ ]:
train_X_feat = np.c_[train_ddq, train_dq, np.sign(train_dq)]
test_X_feat = np.c_[test_ddq, test_dq, np.sign(train_dq)]

In [ ]:
lin_feat = sklearn.linear_model.LinearRegression().fit(train_X_feat, train_y)
lin_feat.coef_, lin_feat.intercept_

In [ ]:
train_y_hat = lin_feat.predict(train_X_feat)
train_rmse = np.sqrt(np.mean((train_y - train_y_hat)**2))
#test_y_hat = test_X @ theta_ls
test_y_hat = lin_feat.predict(test_X_feat)
test_rmse = np.sqrt(np.mean((test_y - test_y_hat)**2))
train_rmse, test_rmse

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
plt.suptitle('Linear IDM with static friction')
ax[0].plot(train_t, train_y, "k", label='true')
ax[0].plot(train_t, train_y_hat, "b", label='predicted')
ax[0].plot(train_t, train_y - train_y_hat, "r", label='prediction error')
ax[0].set_title('train data')
ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('Force (N)')

ax[1].plot(test_t, test_y, "k", label='true')
ax[1].plot(test_t, test_y_hat, "b", label='predicted')
ax[1].plot(test_t, test_y - test_y_hat, "r", label='prediction error')
ax[1].set_title('test data')
ax[1].set_xlabel('Time (s)')
ax[1].set_ylabel('Force (N)');

Train and test performance significantly improved, force estimation qualitatively much better! However, it requires some knowldege.
What if we didn't know about static friction? What if the previous model is still not enough?

# Feed-forward neural network IDM

What if we didn't know about static friction? What if the true torque model is even more complex?

We may fit a neural-netwok IDM!

$$ \tau(t) = \mathrm{FF}(\ddot q, \dot q) $$


In [ ]:
...


## Physics-inspired neural network

What if we trust Newton, but we want some flexibility for / are ignorant about friction? We can design a custom physics-inspired neural network!

$$\tau(t) = M \ddot q + \mathrm{FF}(\dot q)$$

Linear in $\ddot q$, non-linear in $\dot q$

In [ ]:
...